In [1]:
from brainhack.pulse import Tukey
from brainhack.sequence import Sequence, Modulation
from brainhack.system import System
from brainhack.simulator import Simulator
from brainhack.corrector import Corrector, Correctable

import matplotlib.pyplot as plt
import numpy as np

## Setup

In [2]:
pulse = Tukey(
    duration = 1e-3,                    # in s
    shape = .3,
    flipAngle = 299,                    # in degree
    offset = 7000                       # in Hz
)

sequence_noDummy = Sequence(
    modulation = Modulation.BP,
    pulse = pulse,
    N_pulsePerOffset = 1,
    N_pulse = 6,
    N_burst = 10,
    N_adc = 96,
    N_dummyADC = 0,
    dt_interPulse = 1.5e-3,             # in s
    TR_burst = 100e-3,                  # in s
    dt_lastBurst = 9e-3,                # in s
    es = 6e-3,                          # in s
    tr = 20,                            # in s
    readout_flipAngle = 6               # in degree
)

sequence_withDummy = Sequence(
    modulation = Modulation.BP,
    pulse = pulse,
    N_pulsePerOffset = 1,
    N_pulse = 6,
    N_burst = 10,
    N_adc = 96,
    N_dummyADC = 50,
    dt_interPulse = 1.5e-3,             # in s
    TR_burst = 100e-3,                  # in s
    dt_lastBurst = 9e-3,                # in s
    es = 6e-3,                          # in s
    tr = 20,                            # in s
    readout_flipAngle = 6               # in degree
)

system = System(
    pulse = pulse,
    poolFree_M0 = 1,
    poolBound_M0 = 0.1,
    poolFree_T1 = 1,                    # in s
    poolBound_T1 = 1,                   # in s 
    poolBound_T1D = 0.01,               # in s
    poolFree_T2 = 0.1,                  # in s
    poolBound_T2 = 1e-5,                # in s
    poolFreeBound_exchangeRate = 20     # in s^-1
)

simulator_noDummy = Simulator(
    system = system,
    sequence = sequence_noDummy,
    export_readMatrix = True,
)

simulator_withDummy = Simulator(
    system = system,
    sequence = sequence_withDummy,
    export_readMatrix = True,
)

corrector_noDummy = Corrector.Simple(simulator_noDummy)
corrector_withDummy = Corrector.Simple(simulator_withDummy)


## Example run (with dummy ADC)

In [ ]:
out = simulator_withDummy.SteadyState()


for key, value in out.items():
    print(f'{key} | {value.shape}', value, sep='\n', end='\n\n')

print(f"ihMTR = {2*(out['MTs'][0] - out['MTd_CM'][0]) / out['MT0'][0] * 100:.2f}%")

print("MT0 Residuals =", out['MT0'][0] - 0.8062509769138312)
print("MTs Residuals =", out['MTs'][0] - 0.6908388239773408)
print("MTd Residuals =", out['MTd_CM'][0] - 0.5771086868643962)

## Example $\mathrm{B}_1^+$ correction

### Manual

In [ ]:
flipAngle_nominal = pulse.flipAngle
B1rel_range = np.linspace(.1, 1.5, 36)
ihMTR_values = []

for B1rel in B1rel_range:
    pulse.flipAngle = flipAngle_nominal * B1rel
    out = simulator_withDummy.SteadyState()
    ihMTR_temp = 2*(out['MTs'][0] - out['MTd_CM'][0]) / out['MT0'][0] * 100
    ihMTR_values.append(ihMTR_temp)

# reset flip angle to nominal
pulse.flipAngle = flipAngle_nominal

print('B1+ correction Residuals =',
    ihMTR_values - np.array([
    0.04231110571522111, 0.1549421767342752, 0.3978168044444873, 0.8231099305010158,
    1.4710833529953897, 2.3637454392360318, 3.502613753466784, 4.870086771688227,
    6.433298001725599, 8.149181734682612, 9.969670584605636, 11.846282230364976,
    13.733695055594932, 15.592187084929654, 17.389000246394353, 19.098801119382074,
    20.70345766548359, 22.191356546824153, 23.556462375668314, 24.797280935800075,
    25.91584321365408, 26.916783778769798, 27.806550782327648, 28.592757992171826,
    29.28367194511463, 29.8878180260107, 30.413685929748024, 30.86951534458712,
    31.26314501023589, 31.601911281587476, 31.892585210609113, 32.14133958655248,
    32.35373925293102, 32.53474939864668, 32.68875751079569, 32.819605396523635,
    ], dtype=np.float64), sep='\n'
)

### Using Corrector module

### Visualization

In [ ]:
# Tracer ihMTR en fonction de B1rel
plt.plot(B1rel_range*flipAngle_nominal, ihMTR_values, marker='o', linestyle='-', color='r')
plt.xlabel('FAsat (degrés)')
plt.ylabel('ihMTR (%)')
plt.title('ihMTR en fonction de FAsat')
plt.grid(True)
plt.show()

In [ ]:
# Tracer ihMTR en fonction de B1rel
plt.plot(B1rel_range, ihMTR_values, marker='o', linestyle='-', color='r')
plt.xlabel('B1 relatif (B1/B1_nominal)')
plt.ylabel('ihMTR (%)')
plt.title('ihMTR en fonction de B1 relatif')
plt.grid(True)
plt.show()


In [ ]:
(B1rel_range*flipAngle_nominal)[14]

In [ ]:
print(ihMTR_values[0] - 0.04231110571522111)
print(ihMTR_values[14] - 17.389000246394353)
print(ihMTR_values[-1] - 32.819605396523635)

In [ ]:
MT0_noDummy, MTs_noDummy, *MTds_noDummy, read_noDummy = simulator_noDummy.SteadyState().values()
MT0_withDummy, MTs_withDummy, *MTds_withDummy, read_withDummy = simulator_withDummy.SteadyState().values()


In [ ]:
inv_read_withDummy = np.linalg.inv(read_withDummy)
print('MT0')
print(np.linalg.matrix_power(inv_read_withDummy, 3) @ MT0_withDummy)
print(MT0_noDummy)
print(np.linalg.matrix_power(read_noDummy, 3) @ MT0_noDummy)
print(MT0_withDummy)

print('MTs')
print(np.linalg.matrix_power(inv_read_withDummy, 3) @ MTs_withDummy)
print(MTs_noDummy)
print(np.linalg.matrix_power(read_noDummy, 3) @ MTs_noDummy)
print(MTs_withDummy)

print('MTd CM')
print(np.linalg.matrix_power(inv_read_withDummy, 3) @ MTds_withDummy[0])
print(MTds_noDummy[0])
print(np.linalg.matrix_power(read_noDummy, 3) @ MTds_noDummy[0])
print(MTds_withDummy[0])

print('MTd ALT')
print(np.linalg.matrix_power(inv_read_withDummy, 3) @ MTds_withDummy[1])
print(MTds_noDummy[1])
print(np.linalg.matrix_power(read_noDummy, 3) @ MTds_noDummy[1])
print(MTds_withDummy[1])

In [ ]:
allMats = [np.linalg.matrix_power(read_noDummy, i) for i in range(sequence_noDummy.N_adc)]
allMTds = [None, None]
allMTRds = [None, None]
allihMTs = [None, None]
allihMTRs = [None, None]

allMT0 = np.array([mat @ MT0_noDummy for mat in allMats])

allMTs = np.array([mat @ MTs_noDummy for mat in allMats])
allMTRs = np.nan_to_num(100 * (1 - allMTs / allMT0), 0)

allMTds[0] = np.array([mat @ MTds_noDummy[0] for mat in allMats])
allMTRds[0] = np.nan_to_num(100 * (1 - allMTds[0] / allMT0), 0)

allMTds[1] = np.array([mat @ MTds_noDummy[1] for mat in allMats])
allMTRds[1] = np.nan_to_num(100 * (1 - allMTds[1] / allMT0), 0)

allihMTs[0] = 2 * (allMTs - allMTds[0])
allihMTRs[0] = np.nan_to_num(100 * allihMTs[0] / allMT0, 0)

allihMTs[1] = 2 * (allMTs - allMTds[1])
allihMTRs[1] = np.nan_to_num(100 * allihMTs[1] / allMT0, 0)

In [ ]:
x = range(1, sequence_noDummy.N_adc + 1)

fig, axes = plt.subplots(2, 6, sharex=True, figsize=(32, 8))
axes = axes.flatten()

axes[0].set_title("$MT_0$")
axes[0].grid()
axes[0].set_ylim(0, 1)
axes[0].plot(x, allMT0[:,0:2])
axes[0].set_xlim(1, sequence_noDummy.N_adc)
axes[0 + 6].remove()

axes[1].set_title("$MT_s$")
axes[1].grid()
axes[1].set_ylim(0, 1)
axes[1].plot(x, allMTs[:,0:2])
axes[1 + 6].set_title("$MTR_s$")
axes[1 + 6].grid()
axes[1 + 6].set_ylim(0, 40)
axes[1 + 6].plot(x, allMTRs[:,0:2])

axes[2].set_title("$MT_d^{CM}$")
axes[2].grid()
axes[2].set_ylim(0, 1)
axes[2].plot(x, allMTds[0][:,0:2])
axes[2 + 6].set_title("$MTR_d^{CM}$")
axes[2 + 6].grid()
axes[2 + 6].set_ylim(0, 40)
axes[2 + 6].plot(x, allMTRds[0][:,0:2])

axes[3].set_title("$MT_d^{ALT}$")
axes[3].grid()
axes[3].set_ylim(0, 1)
axes[3].plot(x, allMTds[1][:,0:2])
axes[3 + 6].set_title("$MTR_d^{ALT}$")
axes[3 + 6].grid()
axes[3 + 6].set_ylim(0, 40)
axes[3 + 6].plot(x, allMTRds[1][:,0:2])

axes[4].set_title("$ihMT_d^{CM}$")
axes[4].grid()
axes[4].set_ylim(0, .40)
axes[4].plot(x, allihMTs[0][:,0:2])
axes[4 + 6].set_title("$ihMTR_d^{CM}$")
axes[4 + 6].grid()
axes[4 + 6].set_ylim(0, 40)
axes[4 + 6].plot(x, allihMTRs[0][:,0:2])

axes[5].set_title("$ihMT_d^{ALT}$")
axes[5].grid()
axes[5].set_ylim(0, .40)
axes[5].plot(x, allihMTs[1][:,0:2])
axes[5 + 6].set_title("$ihMTR_d^{ALT}$")
axes[5 + 6].grid()
axes[5 + 6].set_ylim(0, 40)
axes[5 + 6].plot(x, allihMTRs[1][:,0:2])

plt.show()

In [ ]:
x = range(1, sequence_noDummy.N_adc + 1)

fig, axes = plt.subplots(1, 2, sharex=True, figsize=(20, 5))
axes = axes.flatten()

axes[0].plot(x, allMT0.T[0], color='C0', linestyle='dotted', label='MT0')
axes[0].plot(x, allMTs.T[0], color='C2', linestyle='-.', label='MTs')
axes[0].plot(x, allMTds[0].T[0], color='k', linestyle='--', label='MTd CM')
axes[0].plot(x, allMTds[1].T[0], color='C3', linestyle='--', label='MTd ALT')
axes[0].plot(x, allihMTs[0].T[0], color='k', linestyle='-', label='ihMT CM')
axes[0].plot(x, allihMTs[1].T[0], color='C3', linestyle='-', label='ihMT ALT')

axes[1].plot(x, allMTRs.T[0], color='C2', linestyle='-.', label='MTRs')
axes[1].plot(x, allMTRds[0].T[0], color='k', linestyle='--', label='MTRd CM')
axes[1].plot(x, allMTRds[1].T[0], color='C3', linestyle='--', label='MTRd ALT')
axes[1].plot(x, .5 * allihMTRs[0].T[0], color='k', linestyle='-', label='ihMTR CM')
axes[1].plot(x, .5 * allihMTRs[1].T[0], color='C3', linestyle='-', label='ihMTR ALT')

axes[0].legend()
axes[0].grid()
axes[0].set_xlabel('# adc')
axes[0].set_xlim(1, sequence_noDummy.N_adc)
axes[0].set_ylabel('(ih)MT [a.u.]')
axes[0].set_ylim(0, 1)

axes[1].legend()
axes[1].grid()
axes[1].set_xlabel('# adc')
axes[1].set_xlim(1, sequence_noDummy.N_adc)
axes[1].set_ylabel('(ih)MTR [%]')
axes[1].set_ylim(0, 40)

plt.show()

